# Suite BLW — A store given a store below

A store may be given the store below it, from which it loads what it is asked for, when it is asked ([CAP-283: a store given a store below](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-283), [CAP-284: objects loaded on demand](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-284)): `Builders.Store(below)`. The store below gives an object's document, by its path (`document(path)`): a one-transaction document of its properties and its entries, each object they link by its path. The store above loads an object the first time its values or entries are read, as its committed data, observed by none; an object is present in it, made in it or loaded into it, or it is not: an entry that links one not present holds it by its path, which reading the link loads, and a path the store below holds no object at resolves to none. Given no store below, a store holds what is written to it, as it always has. A transaction of the store above is written through, as plain data, before it commits: the store below commits it first, and a refusal there refuses it; and what the store below commits, the store above applies to the objects present in it alone, in the order the store below committed it, loading nothing ([CAP-285: writes through, others' writes in](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-285)).


In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders as B, Changes, Plain, References
from support import raises


def crm(txn):
    """crm.Person, owned by the singleton book by key, with a name and a home; persons know persons; a book has one lead; persons mentor persons."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Home": S.OfObject, "Holds": S.OfRelation, "Knows": S.OfRelation, "Book": S.OfObject,
                  "Person": S.OfObject, "Keeps": S.OfRelation, "Pet": S.OfObject, "Walks": S.OfRelation, "Introduces": S.OfRelation,
                  "Leads": S.OfRelation, "Mentors": S.OfRelation})
        .spec([
            lambda _: _.Home.name("crm.Home").properties({"city": S.OfStr, "zip": S.OfStr}),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")])
            .properties({"key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]),
            lambda _: _.Keeps.name("crm.Keeps").links([lambda t: t.name("keeper"), lambda t: t.name("pet")])
            .properties({"name": S.OfStr}).uniques([["keeper", "name"], ["pet"]]),
            lambda _: _.Introduces.name("crm.Introduces").links([lambda t: t.name("host"), lambda t: t.name("guest"),
                                                                 lambda t: t.name("friend")]),
            lambda _: _.Leads.name("crm.Leads").links([lambda t: t.name("book"), lambda t: t.name("lead")])
            .uniques([["lead"]]),  # a book's one lead
            lambda _: _.Mentors.name("crm.Mentors").links([lambda t: t.name("mentor"), lambda t: t.name("mentee")]),
            lambda _: _.Walks.name("crm.Walks").links([lambda t: t.name("walker"), lambda t: t.name("walked")]),
            lambda _: _.Pet.name("crm.Pet").ref(True).properties({"kind": S.OfStr}).adjacencies([
                lambda t: t.name("keeper").relation([_.Keeps]).me("pet").owner(True),
                lambda t: t.name("walkers").relation([_.Walks]).me("walked")]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book"),
                "leads": lambda t: t.relation([_.Leads]).me("book")}),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr, "home": _.Home}).adjacencies([
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.name("known_by").relation([_.Knows]).me("known"),
                lambda t: t.name("pets").relation([_.Keeps]).me("keeper"),
                lambda t: t.name("walks").relation([_.Walks]).me("walker"),
                lambda t: t.name("mentors").relation([_.Mentors]).me("mentor"),  # a mentee's side has no adjacency
                lambda t: t.name("hosts").relation([_.Introduces]).me("host"),
                lambda t: t.name("guested").relation([_.Introduces]).me("guest"),
                lambda t: t.name("befriended").relation([_.Introduces]).me("friend"),
            ]),
        ])]))])


below = B.Store()
below.transaction(crm)
book = below.singleton("crm.TheBook")
with below.transaction():
    people = {n: below.crm.Person(lambda p, n=n: p.book(lambda a: a.insert(lambda e: e.key(n).book(book))).name(n))
              for n in ("ada", "bob", "cy")}
    below.crm.Person(people["ada"], lambda p: p.home(lambda h: h.city("Paris").zip("75001"))
                     .knows(lambda a: a.insert(lambda e: e.known(people["bob"]))))
    rex_below = below.crm.Pet(lambda p: p.keeper(lambda a: a.insert(lambda e: e.name("rex").keeper(people["ada"])))
                              .kind("dog"))
    below.crm.Person(people["bob"], lambda p: p.walks(lambda a: a.insert(lambda e: e.walked(rex_below)))
                     .hosts(lambda a: a.insert(lambda e: e.guest(people["ada"]).friend(people["cy"]))))
above = B.Store(below)
above.transaction(crm)


def loaded():
    """How many objects of the store below's the store above has loaded."""
    return len(above._loaded)

<a id="BLW-01"></a>
## BLW-01 · Nothing is loaded until it is read; reading an object loads it alone, an entry linking one not yet loaded holding it by its path until the link is read

In [ ]:
ours = above.singleton("crm.TheBook")
before = loaded()
keys = [e.key.get() for e in ours.contacts]
after_book = loaded()
ada = next(e.contact for e in ours.contacts if e.key.get() == "ada")
name = ada.name.get()
assert json.dumps([before, keys, after_book, name, loaded()]) == '[0, ["ada", "bob", "cy"], 1, "ada", 2]'

<a id="BLW-02"></a>
## BLW-02 · An object is the same object each time it is named, through any entry; its value objects and entries are loaded with it; an object reached by its reference, or through an entry, loads alone, its owner not loaded, its reference through its owner's path

In [ ]:
bob = next(e.contact for e in ours.contacts if e.key.get() == "bob")
assert next(e.known for e in ada.knows) is bob and next(e.book for e in ada.book) is ours
assert next(e.knower for e in bob.known_by) is ada
assert json.dumps([ada.home.city.get(), ada.home.zip.get(), bob.name.get(), ada.as_schema().name]) == '["Paris", "75001", "bob", "crm.Person"]'
fresh = B.Store(below)
fresh.transaction(crm)
rex_path = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "ada"},
            "next": {"entry": {"i": "pets", "o": "keeper", "properties": {"name": "rex"}, "next": {"target": {}}}}}}}
rex = References.resolve(fresh, Plain.FromPlain(fresh)(fresh.registered("References.OfObject"), {"path": rex_path}))
rex_reference = JSON.dumps(Plain.ToPlain(fresh)(fresh.registered("References.OfObject"), References.of(fresh, rex)))
assert rex_reference == JSON.dumps({"path": rex_path})  # its owner not present
assert json.dumps([len(fresh._loaded), repr(rex), rex.kind.get(), len(fresh._loaded)]) == '[1, "<crm.Pet>", "dog", 1]'
other = B.Store(below)
other.transaction(crm)
walker = next(e.contact for e in other.singleton("crm.TheBook").contacts if e.key.get() == "bob")
walked = next(e.walked for e in walker.walks)
assert json.dumps([len(other._loaded), repr(walked), walked.kind.get(), len(other._loaded)]) == '[3, "<crm.Pet>", "dog", 3]'

<a id="BLW-03"></a>
## BLW-03 · The store below gives an object's document by its path, each object it links by its path; none where it holds none, where a path above resolves to no object

In [ ]:
path = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "cy"},
                                                       "next": {"target": {}}}}}
document = below.document(path)
nobody = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book",
                                                          "properties": {"key": "zed"}, "next": {"target": {}}}}}
assert json.dumps([JSON.dumps(document), below.document(nobody)]) == '["{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"0\\": {\\"schema\\": [{\\"$ref\\": {\\"singleton\\": \\".\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"crm\\"}, \\"next\\": {\\"entry\\": {\\"i\\": \\"schemas\\", \\"o\\": \\"\\", \\"properties\\": {\\"name\\": \\"Person\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}, \\"spec\\": [{\\"0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"cy\\"}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"book\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"book\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}}]}}, {\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"cy\\"}}]}}]}]}}]}}, {\\"as_adjacency\\": {\\"name\\": \\"befriended\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"host\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"bob\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}, {\\"as_link\\": {\\"name\\": \\"guest\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"ada\\"}, \\"next\\": {\\"target\\": {}}}}}}}}]}}]}]}}]}}]}]}]}}]}", null]'
reference = Plain.FromPlain(above)(above.registered("References.OfObject"), {"path": nobody})
assert References.resolve(above, reference) is None

<a id="BLW-04"></a>
## BLW-04 · Loading is no transaction of the store's: its followers are given nothing of it, and a transaction open while an object loads keeps its own changes alone

In [ ]:
effects = []
end = Changes.observe(above, effects.append)
cy = next(e.contact for e in ours.contacts if e.key.get() == "cy")
with above.transaction():
    seen = cy.name.get()  # loaded within the transaction
    above.crm.Person(cy, lambda p: p.name("Cy"))
end()
(effect,) = effects
assert json.dumps([seen, cy.name.get(), JSON.dumps(Plain.ToPlain(above).Transaction(effect))]) == '["cy", "Cy", "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"cy\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"o0\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"Cy\\"}}]}}]}]}]}}]}"]'


<a id="BLW-05"></a>
## BLW-05 · An object loaded has its reference as the store below has it; given no store below, a store holds what is written to it

In [ ]:
alone = B.Store()
alone.transaction(crm)
with alone.transaction():
    alone.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("dee").book(alone.singleton("crm.TheBook")))))
def path_of(store, x):
    """An object's reference, as plain data."""
    return JSON.dumps(Plain.ToPlain(store)(store.registered("References.OfObject"), References.of(store, x)))


assert json.dumps([path_of(above, cy) == path_of(below, people["cy"]),
                   [e.key.get() for e in alone.singleton("crm.TheBook").contacts], alone._below]) == '[true, ["dee"], null]'

<a id="BLW-06"></a>
## BLW-06 · An object the store above makes is found again by its path, and one the store below adds after its owner is loaded above is loaded when it is named

In [ ]:
with above.transaction():
    eve = above.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("eve").book(ours))).name("eve"))
found = References.resolve(above, References.of(above, eve))
with below.transaction():
    below.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("fay").book(book))).name("fay"))
fay_path = {"singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": "fay"},
                                                           "next": {"target": {}}}}}
fay = References.resolve(above, Plain.FromPlain(above)(above.registered("References.OfObject"), {"path": fay_path}))
assert json.dumps([found is eve, fay.name.get()]) == '[true, "fay"]'

<a id="BLW-07"></a>
## BLW-07 · A transaction above that removes the owning entry of an object not present writes it by its owner's key, loading nothing, and removes, among the objects present, every entry linking it; one that removes an entry linking an object not present, which it names, loads it; a link read within a transaction is resolved there too

In [ ]:
with below.transaction():  # a contact the store above will remove, and one who knows it
    gus = below.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("gus").book(book))).name("gus"))
    below.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("hal").book(book))).name("hal")
                     .knows(lambda a: a.insert(lambda e: e.known(gus))))
third = B.Store(below)
third.transaction(crm)
books = third.singleton("crm.TheBook")
hal = next(x.contact for x in books.contacts if x.key.get() == "hal")
knew = len(list(hal.knows))
effects7 = []
Changes.observe(third, effects7.append)
with third.transaction():
    third.crm.Book(books, lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("gus"))))  # gus not present
    within = next(x.contact for x in books.contacts if x.key.get() == "ada").name.get()  # ada loaded, within it
removal = JSON.dumps(Plain.ToPlain(third).Transaction(effects7[-1]))
ada3 = next(x.contact for x in books.contacts if x.key.get() == "ada")
loaded_before = len(third._loaded)
with third.transaction():
    third.crm.Person(ada3, lambda p: p.knows(lambda a: a.clear()))  # ada knew bob, not present
unknowing = JSON.dumps(Plain.ToPlain(third).Transaction(effects7[-1]))
assert json.dumps([knew, len(list(hal.knows)), within, removal, len(third._loaded) - loaded_before, unknowing]) == '[1, 0, "ada", "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}}, \\"spec\\": [{\\"o0\\": [{\\"as_adjacency\\": {\\"name\\": \\"contacts\\", \\"mutations\\": [{\\"remove\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"gus\\"}}]}}]}]}}]}}]}]}]}}]}", 1, "{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"bob\\"}, \\"next\\": {\\"target\\": {}}}}}}}, \\"o1\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"ada\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"o1\\": [{\\"as_adjacency\\": {\\"name\\": \\"knows\\", \\"mutations\\": [{\\"remove\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"known\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"symbol\\": \\"o0\\"}}}]}}]}]}}]}}]}]}]}}]}"]'


<a id="BLW-08"></a>
## BLW-08 · Stores stack: a store above a store above is given documents in which the objects not present in the middle one are by their paths; an entry of three links, one of them read, holds the other by its path still

In [ ]:
middle = B.Store(below)
middle.transaction(crm)
top = B.Store(middle)
top.transaction(crm)
host = next(e.contact for e in top.singleton("crm.TheBook").contacts if e.key.get() == "bob")
(introduction,) = host.hosts
guest = introduction.guest.name.get()
still_by_path = type(introduction._links["friend"]).__name__
assert json.dumps([guest, still_by_path, introduction.friend.name.get(), len(middle._loaded), len(top._loaded)]) == '["ada", "_Unresolved", "Cy", 4, 4]'

<a id="BLW-09"></a>
## BLW-09 · A transaction above is written through: the store below commits it first, within it, and a refusal there refuses it; what another wrote below meanwhile is applied once it ends, in the order the store below committed it

In [ ]:
effects9 = []
end9 = Changes.observe(above, effects9.append)
with above.transaction():  # the store below commits it first
    above.crm.Person(bob, lambda p: p.name("Bob"))
    above.crm.Book(ours, lambda b: b.leads(lambda a: a.insert(lambda e: e.lead(ada))))
written = [people["bob"].name.get(), [e.lead.name.get() for e in book.leads], len(effects9)]
with raises(ValueError, match="breaks unique"):
    with above.transaction():  # ada's lead given to bob above, while another gives it to cy below
        above.crm.Book(ours, lambda b: b.leads(lambda a: a.remove(lambda e: e.lead(ada)).insert(lambda e: e.lead(bob))))
        with below.transaction():
            below.crm.Book(book, lambda b: b.leads(lambda a: a.remove(lambda e: e.lead(people["ada"]))
                                                   .insert(lambda e: e.lead(people["cy"]))))
end9()
assert json.dumps([written, len(effects9), [e.lead.name.get() for e in ours.leads], [e.lead.name.get() for e in book.leads]]) == '[["Bob", ["ada"], 1], 2, ["Cy"], ["Cy"]]'


<a id="BLW-10"></a>
## BLW-10 · What another writes below is applied above to the objects present alone, loading nothing: an object it made, or one not present, is linked by its path; the followers above are told, and so is a store above that one

In [ ]:
effects10 = []
end10 = Changes.observe(above, effects10.append)
before10 = len(above._loaded)
with below.transaction():  # ada renamed, present above; hal renamed, not present; joy made, knowing ada
    below.crm.Person(people["ada"], lambda p: p.name("Ada"))
    below.crm.Person(next(e.contact for e in book.contacts if e.key.get() == "hal"), lambda p: p.name("Hal"))
    below.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("joy").book(book))).name("joy")
                     .knows(lambda a: a.insert(lambda e: e.known(people["ada"]))))
end10()
applied = [JSON.dumps(Plain.ToPlain(above).Transaction(effect)) for effect in effects10]
loaded10 = len(above._loaded) - before10
(knowing,) = ada.known_by
by_path = type(knowing._links["knower"]).__name__
assert json.dumps([ada.name.get(), applied, loaded10, by_path, knowing.knower.name.get(), len(above._loaded) - before10,
                   introduction.guest.name.get()]) == '["Ada", ["{\\"mutations\\": [{\\"upsert\\": {\\"exists\\": [{\\"symbols\\": {\\"o0\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"joy\\"}, \\"next\\": {\\"target\\": {}}}}}}}, \\"o1\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"target\\": {}}}}}, \\"o2\\": {\\"reference\\": {\\"path\\": {\\"singleton\\": \\"crm.TheBook\\", \\"next\\": {\\"entry\\": {\\"i\\": \\"contacts\\", \\"o\\": \\"book\\", \\"properties\\": {\\"key\\": \\"ada\\"}, \\"next\\": {\\"target\\": {}}}}}}}}, \\"spec\\": [{\\"o1\\": [{\\"as_adjacency\\": {\\"name\\": \\"contacts\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"contact\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"symbol\\": \\"o0\\"}}}]}}, {\\"as_property\\": {\\"name\\": \\"key\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"joy\\"}}]}}]}]}}]}}]}, {\\"o2\\": [{\\"as_adjacency\\": {\\"name\\": \\"known_by\\", \\"mutations\\": [{\\"insert\\": {\\"entries\\": [{\\"mutations\\": [{\\"as_link\\": {\\"name\\": \\"knower\\", \\"mutations\\": [{\\"set\\": {\\"ref\\": {\\"symbol\\": \\"o0\\"}}}]}}]}]}}]}}]}, {\\"o2\\": [{\\"as_property\\": {\\"name\\": \\"name\\", \\"mutations\\": [{\\"set\\": {\\"value\\": \\"Ada\\"}}]}}]}]}]}}]}"], 0, "_Unresolved", "joy", 1, "Ada"]'


<a id="BLW-11"></a>
## BLW-11 · An object another deletes below goes above with every entry linking it; an entry read after, held by no object, links what the store below still holds, and refuses a link to what it no longer holds

In [ ]:
hal_entry = next(e for e in ours.contacts if e.key.get() == "hal")  # hal not present above
(walking,) = bob.walks  # bob walks rex, not present above
by_path11 = [type(hal_entry._links["contact"]).__name__, type(walking._links["walked"]).__name__]
with below.transaction():  # hal removed; bob walks rex no more
    below.crm.Book(book, lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("hal"))))
    below.crm.Person(people["bob"], lambda p: p.walks(lambda a: a.clear()))
with raises(LookupError, match="the store below holds no object at this path"):
    hal_entry.contact
assert json.dumps([by_path11, sorted(e.key.get() for e in ours.contacts), len(list(bob.walks)),
                   walking.walked.kind.get()]) == '[["_Unresolved", "_Unresolved"], ["ada", "bob", "cy", "eve", "fay", "joy"], 0, "dog"]'


<a id="BLW-12"></a>
## BLW-12 · An entry a transaction rewrites, linking an object not present, links it when it loads within the transaction, in the transaction's tables alone, as one loaded in a transaction that writes nothing linking it does; written through, the store below holds the entry as rewritten

In [ ]:
fourth = B.Store(below)
fourth.transaction(crm)
ada4 = next(e.contact for e in fourth.singleton("crm.TheBook").contacts if e.key.get() == "ada")


def befriend(p):
    """Ada made the friend of each introduction she is the guest of."""
    for e in p.guested:
        e.friend(ada4)
    return p


with fourth.transaction():
    fourth.crm.Person(ada4, befriend)
    (rewritten,) = ada4.guested
    by_path12 = type(rewritten._links["host"]).__name__
    host12 = rewritten.host.name.get()  # bob loaded within it
bob4 = rewritten.host
with fourth.transaction():  # one that writes nothing an object loaded within it links
    named12 = next(e.contact for e in fourth.singleton("crm.TheBook").contacts if e.key.get() == "eve").name.get()
assert json.dumps([named12, by_path12, host12, [(e.host.name.get(), e.friend.name.get()) for e in ada4.guested],
                   [(e.guest.name.get(), e.friend.name.get()) for e in bob4.hosts],
                   [(e.host.name.get(), e.friend.name.get()) for e in people["ada"].guested]]) == '["eve", "_Unresolved", "Bob", [["Bob", "Ada"]], [["Ada", "Ada"]], [["Bob", "Ada"]]]'


<a id="BLW-13"></a>
## BLW-13 · An entry another writes below, linking an object present above whose schema has no adjacency to it by name and one not present, is refused where it is applied: name one

In [ ]:
with raises(ValueError, match="name one"):
    with below.transaction():  # kim, made, mentors ada, present above, who has no adjacency to it by name
        below.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("kim").book(book)))
                         .mentors(lambda a: a.insert(lambda e: e.mentee(people["ada"]))))
